In [24]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.kernel_approximation import Nystroem

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import root_mean_squared_error, mean_absolute_error

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv("/Users/nasy/Documents/ml_lr/ml_labs_mai/Regression/data/winequality_full.csv")

df.shape

(6497, 13)

In [4]:
df.columns = (df.columns.str.replace(" ", "_", regex=False))

df.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [5]:
df = df.drop_duplicates(keep='first')
df.shape

(5320, 13)

In [6]:
df["wine_type_binary"] = df["wine_type"].map({
    "red": 0,
    "white": 1
})

In [7]:
X = df.drop(columns=["quality", "wine_type"]).copy()
y = df["quality"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(4256, 12) (1064, 12)
(4256,) (1064,)


# Linear Regression

## не учитываем тип вина как признак

In [8]:
features_no_type = X_train.columns.drop('wine_type_binary')

linear_no_type = LinearRegression()
linear_no_type.fit(X_train[features_no_type], y_train)
linear_no_type_pred = linear_no_type.predict(X_test[features_no_type])

linear_no_type_rmse = root_mean_squared_error(y_test, linear_no_type_pred)
linear_no_type_mae = mean_absolute_error(y_test, linear_no_type_pred)

print("RMSE:", linear_no_type_rmse)
print("MAE:", linear_no_type_mae)

RMSE: 0.733547913328391
MAE: 0.5673044388297395


## учитываем тип вина как признак

In [9]:
linear = LinearRegression()

linear.fit(X_train, y_train)

linear_pred = linear.predict(X_test)
linear_rmse = root_mean_squared_error(y_test, linear_pred)
linear_mae = mean_absolute_error(y_test, linear_pred)

print("RMSE:", linear_rmse)
print("MAE:", linear_mae)

RMSE: 0.729242043708644
MAE: 0.5660009475168252


## разделяем на 2 модели по виду вина

Были построены две отдельные линейные модели для красного и белого вина. Такой подход позволяет коэффициентам признаков различаться в зависимости от типа вина и проверяет гипотезу о том, что зависимости между характеристиками и качеством могут отличаться для двух типов вина

In [10]:
features_no_type = X_train.columns.drop("wine_type_binary")

red_train = X_train["wine_type_binary"] == 0
red_test = X_test["wine_type_binary"] == 0

white_train = X_train["wine_type_binary"] == 1
white_test = X_test["wine_type_binary"] == 1

#red
red_model = LinearRegression()
red_model.fit(X_train.loc[red_train, features_no_type], y_train.loc[red_train])
red_pred = red_model.predict(X_test.loc[red_test, features_no_type])

#white
white_model = LinearRegression()
white_model.fit(X_train.loc[white_train, features_no_type], y_train.loc[white_train])
white_pred = white_model.predict(X_test.loc[white_test, features_no_type])

two_linear_pred = pd.Series(
    index=y_test.index,
    dtype=float
)

two_linear_pred.loc[X_test.loc[red_test].index] = red_pred
two_linear_pred.loc[X_test.loc[white_test].index] = white_pred


two_linear_rmse = root_mean_squared_error(y_test, two_linear_pred)
two_linear_mae = mean_absolute_error(y_test, two_linear_pred)

print("RMSE:", two_linear_rmse)
print("MAE:", two_linear_mae)

RMSE: 0.7311777550275584
MAE: 0.5649656064947207


## Ridge

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)

ridge_rmse = root_mean_squared_error(y_test, ridge_pred)
ridge_mae = mean_absolute_error(y_test, ridge_pred)

print("RMSE:", ridge_rmse)
print("MAE:", ridge_mae)

RMSE: 0.7292529954772714
MAE: 0.5660054010796398


## Lasso

In [13]:
lasso = Lasso(alpha=0.001, max_iter=10000)

lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)

lasso_rmse = root_mean_squared_error(y_test, lasso_pred)
lasso_mae = mean_absolute_error(y_test, lasso_pred)

print("RMSE:", lasso_rmse)
print("MAE:", lasso_mae)

RMSE: 0.7294847358833059
MAE: 0.5661517888064503


## Spline + Ridge

оставляем исходные признаки и дополнительно добавляем spline-представление для наиболее важных признаков

In [14]:
spline_features = [
    "alcohol",
    "volatile_acidity",
    "density",
    "sulphates"
]

spline_scaler = StandardScaler()

X_train_original = spline_scaler.fit_transform(X_train)
X_test_original = spline_scaler.transform(X_test)

spline = SplineTransformer(
    n_knots=5,
    degree=1,
    knots="quantile",
    include_bias=False,
    extrapolation="linear"
)

X_train_spline = spline.fit_transform(X_train[spline_features])
X_test_spline = spline.transform(X_test[spline_features])

X_train_spline_full = np.column_stack([X_train_original, X_train_spline])
X_test_spline_full = np.column_stack([X_test_original, X_test_spline])

spline_ridge = Ridge(alpha=10)
spline_ridge.fit(X_train_spline_full, y_train)
spline_pred = spline_ridge.predict(X_test_spline_full)

spline_rmse = root_mean_squared_error(y_test, spline_pred)
spline_mae = mean_absolute_error(y_test, spline_pred)

print("RMSE:", spline_rmse)
print("MAE:", spline_mae)

RMSE: 0.7246960363673558
MAE: 0.5644930032527572


## RBF + Ridge

In [15]:
rbf_scaler = StandardScaler()
X_train_rbf_scaled = rbf_scaler.fit_transform(X_train)
X_test_rbf_scaled = rbf_scaler.transform(X_test)

rbf = Nystroem(
    kernel="rbf",
    gamma=0.03,
    n_components=1200,
    random_state=42,
    n_jobs=-1
)

X_train_rbf = rbf.fit_transform(X_train_rbf_scaled)
X_test_rbf = rbf.transform(X_test_rbf_scaled)

rbf_ridge = Ridge(alpha=0.1)
rbf_ridge.fit(X_train_rbf, y_train)
rbf_pred = rbf_ridge.predict(X_test_rbf)

rbf_rmse = root_mean_squared_error(y_test,rbf_pred)
rbf_mae = mean_absolute_error(y_test, rbf_pred)

print("RMSE:", rbf_rmse)
print("MAE:", rbf_mae)

RMSE: 0.697308875744542
MAE: 0.5422309586132954


# Boosting

## sklearn

In [19]:
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42
)

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

gb_rmse = root_mean_squared_error(y_test, gb_pred)
gb_mae = mean_absolute_error(y_test, gb_pred)

print("RMSE:", gb_rmse)
print("MAE:", gb_mae)

RMSE: 0.6871699201535585
MAE: 0.5337214796809506


## catboost

In [21]:
cat = CatBoostRegressor(
    depth=8,
    learning_rate=0.02,
    iterations=800,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

cat.fit(X_train, y_train)
cat_pred = cat.predict(X_test)

cat_rmse = root_mean_squared_error(y_test, cat_pred)
cat_mae = mean_absolute_error(y_test, cat_pred)

print("RMSE:", cat_rmse)
print("MAE:", cat_mae)

RMSE: 0.6834061177788844
MAE: 0.5283711801245797


## xgboost

In [22]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

xgb_rmse = root_mean_squared_error(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print("RMSE:", xgb_rmse)
print("MAE:", xgb_mae)

RMSE: 0.6892111897468567
MAE: 0.5307540893554688


## lightgbm

In [23]:
lgbm = LGBMRegressor(
    objective="regression",
    num_leaves=63,
    learning_rate=0.02,
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm.fit(X_train, y_train)

lgbm_pred = lgbm.predict(X_test)

lgbm_rmse = root_mean_squared_error(y_test, lgbm_pred)
lgbm_mae = mean_absolute_error(y_test, lgbm_pred)

print("RMSE:", lgbm_rmse)
print("MAE:", lgbm_mae)

RMSE: 0.6924722812630467
MAE: 0.5343652511787624
